![Slide 1](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/1.svg)

![Slide 2](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/2.svg)

![Slide 3](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/3.svg)

![Slide 4](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/4.svg)

![Slide 5](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/5.svg)

![Slide 6](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/6.svg)

![Slide 7](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/7.svg)

In [ ]:
# Creating a ToolCall object (ch02 example 1)
from llm_agents_from_scratch.data_structures.tool import ToolCall

croissant_tool_call = ToolCall(
    tool_name="web-search-tool",
    arguments={
        "query": "Croissant bakeries in New York City and their prices.",
    },
)
croissant_tool_call

In [ ]:
# Creating a ToolCallResult (ch02 example 2)
from llm_agents_from_scratch.data_structures.tool import ToolCallResult

result = ToolCallResult(
    tool_call_id=croissant_tool_call.id_,
    content={
        "search_results": {
            "hits": [
                {"data": "..."},
            ],
        },
    },
)
result

In [ ]:
# The Hailstone Tool — subclassing BaseTool (ch02 example 3)
from typing import Any

from llm_agents_from_scratch.base.tool import BaseTool
from llm_agents_from_scratch.data_structures.tool import (
    ToolCall,
    ToolCallResult,
)


class Hailstone(BaseTool):
    """A tool that performs a single Hailstone step."""

    @property
    def name(self) -> str:
        """Tool name."""
        return "hailstone"

    @property
    def description(self) -> str:
        """Tool description."""
        return "A tool that performs a Hailstone step on a given input number."

    @property
    def parameters_json_schema(self) -> dict[str, Any]:
        """JSON schema for tool parameters."""
        return {
            "type": "object",
            "properties": {
                "x": {
                    "type": "number",
                    "description": "The input number.",
                },
            },
            "required": ["x"],
        }

    def __call__(
        self,
        tool_call: ToolCall,
        *args: Any,
        **kwargs: Any,
    ) -> ToolCallResult:
        """Execute the tool call."""
        x = tool_call.arguments.get("x")
        result = x // 2 if x % 2 == 0 else x * 3 + 1
        return ToolCallResult(
            tool_call_id=tool_call.id_,
            content=result,
            error=False,
        )

In [ ]:
# Invoking the Hailstone tool
hailstone_tool = Hailstone()

tool_call = ToolCall(tool_name="hailstone", arguments={"x": 3})
tool_call_result = hailstone_tool(tool_call)
print(tool_call_result)

In [ ]:
# The same thing with SimpleFunctionTool (ch02 example 5)
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool


def hailstone_step_func(x: int) -> int:
    """Performs a single step of the Hailstone sequence."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


hailstone_tool = SimpleFunctionTool(hailstone_step_func)

print(f"Name:        {hailstone_tool.name}")
print(f"Description: {hailstone_tool.description}")
print(f"Schema:      {hailstone_tool.parameters_json_schema}")

In [ ]:
# Invoking the SimpleFunctionTool (ch02 example 6)
from llm_agents_from_scratch.data_structures import ToolCall

tool_call = ToolCall(tool_name="hailstone_fn", arguments={"x": 3})
res = hailstone_tool(tool_call)
res

![Slide 8](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/8.svg)

In [ ]:
# Instantiate an OllamaLLM (ch03 example 2)
from llm_agents_from_scratch.llms.ollama import OllamaLLM

llm = OllamaLLM(model="glm-5.1:cloud")

In [ ]:
# Simple completion (ch03 example 3)
response = await llm.complete("Tell me a joke.")
print(response)

In [ ]:
# Structured output (ch03 example 4)
from typing import Literal

from pydantic import BaseModel


class Joke(BaseModel):
    """A structured output model for Jokes."""

    subject: Literal["math", "physics", "biology"]
    joke: str


joke = await llm.structured_output(prompt="Tell me a joke.", mdl=Joke)
print(joke.__class__.__name__)
print(joke)

![Slide 9](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/9.svg)

In [ ]:
# Eliciting a tool call (ch03 example 5)
from llm_agents_from_scratch.tools import SimpleFunctionTool


def hailstone_step_func(x: int) -> int:
    """Performs a single step of the Hailstone sequence."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


hailstone_tool = SimpleFunctionTool(hailstone_step_func)

user_msg, response_msg = await llm.chat(
    "What is the result of taking the next step of the "
    "Hailstone sequence on the number 3?\n\n"
    "Be very succinct in your response.",
    tools=[hailstone_tool],
)
print(response_msg.tool_calls)

In [ ]:
# Executing the tool call
tool_call = response_msg.tool_calls[0]
tool_call_result = hailstone_tool(tool_call)
print(tool_call_result)

In [ ]:
# Sending tool results back to the LLM (ch03 closing the loop)
tools_msg, final_response = await llm.continue_chat_with_tool_results(
    tool_call_results=[tool_call_result],
    chat_history=[user_msg, response_msg],
)
print(final_response)

![Slide 10](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/10.svg)

![Slide 11](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/11.svg)

In [ ]:
# Enable logging so we can see the agent's steps
import logging

from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

In [ ]:
# Define the Hailstone tool (obfuscated to force tool use) (ch04 example 3)
from pydantic import BaseModel

from llm_agents_from_scratch.tools import PydanticFunctionTool


class AlgoParams(BaseModel):
    """Params for next_number."""

    x: int


def next_number(params: AlgoParams) -> int:
    """Generate the next number of the sequence."""
    if params.x % 2 == 0:
        return params.x // 2
    return 3 * params.x + 1


tool = PydanticFunctionTool(next_number)

In [ ]:
# Define the LLM and agent (ch04 example 3)
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.llms import OllamaLLM

llm = OllamaLLM(model="glm-5.1:cloud")
llm_agent = LLMAgent(llm=llm, tools=[tool])

In [ ]:
# Define and run the task (ch04 example 3)
from llm_agents_from_scratch.data_structures import Task

instruction_template = """
You are given a tool, `next_number`, that generates the next number in the
sequence given the current number.

Start with the number x={x}.

<rules>
CALL `next_number` on the current number x
STOP AND WAIT for the result.
REPEAT this step-by-step process until the number 1 is reached.
FINAL RESULT: When you receive the number 1, provide the complete sequence you
observed from start to finish (including the starting number x and ending number
1).
</rules>

<warnings>
NEVER fabricate or simulate tool call results
NEVER make multiple tool calls in one response
STOP and WAIT - ALWAYS wait for the actual tool response before deciding next
steps
</warnings>
""".strip()

number = 6
task = Task(instruction=instruction_template.format(x=number))
handler = llm_agent.run(task, max_steps=15)

In [ ]:
# Inspect the results
print(f"Done: {handler.done()}")
print(f"Steps taken: {handler.step_counter}")

result = (
    handler.result() if not handler.exception() else str(handler.exception())
)
print(f"\nResult: {result}")

In [ ]:
# The rollout — every step the agent took
print(handler.rollout)

In [ ]:
import asyncio

# Concurrent runs — multiple tasks on the same agent
task_a = Task(instruction=instruction_template.format(x=3))
task_b = Task(instruction=instruction_template.format(x=7))

handler_a = llm_agent.run(task_a, max_steps=20)
handler_b = llm_agent.run(task_b, max_steps=20)

# Both are running concurrently on the event loop
await asyncio.gather(handler_a.background_task, handler_b.background_task)

print(f"Task A steps: {handler_a.step_counter}")
print(f"Task B steps: {handler_b.step_counter}")
print(f"\nTask A result: {handler_a.result()}")
print(f"\nTask B result: {handler_b.result()}")

![Slide 12](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/12.svg)

![Slide 13](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/13.svg)

![Slide 14](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/14.svg)

![Slide 15](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/15.svg)

In [ ]:
# Creating an MCPToolProvider (ch05 example 1)
from pathlib import Path

from mcp import StdioServerParameters

from llm_agents_from_scratch.tools.mcp import MCPToolProvider

server_path = Path.cwd().parent / "extra/mcp-hailstone"
server_params = StdioServerParameters(
    command="uv",
    args=["run", "--with", "mcp", "mcp", "run", "main.py"],
    cwd=server_path,
)
provider = MCPToolProvider(
    name="hailstone",
    stdio_params=server_params,
)

In [ ]:
# Discovering tools (ch05 example 3)
tools = await provider.get_tools()
print(f"Number of tools discovered: {len(tools)}")

In [ ]:
# Printing the MCP tool's attributes (ch05 example 5)
print(f"Tool name: {tools[0].name}")
print(f"Tool description: {tools[0].description}")
print(f"Tool parameters: {tools[0].parameters_json_schema}")
print(f"Provider reference: {tools[0].provider}")

![Slide 16](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/16.svg)

In [ ]:
# MCP tool execution (ch05 example 6)
from llm_agents_from_scratch.data_structures import ToolCall

tool_call = ToolCall(
    tool_name=tools[0].name,
    arguments={"x": 5},
)
result = await tools[0](tool_call)
result

In [ ]:
# Real-world MCP: GitHub MCP Server (from more-examples/ch05)
# Unlike stdio, GitHub's MCP server uses streamable HTTP transport.
import getpass
import os

os.environ["GITHUB_PAT"] = getpass.getpass("GitHub PAT: ")

from llm_agents_from_scratch.tools import MCPToolProvider

github_mcp_provider = MCPToolProvider(
    name="github_mcp",
    streamable_http_url="https://api.githubcopilot.com/mcp/",
    streamable_http_headers={
        "Authorization": f"Bearer {os.environ['GITHUB_PAT']}",
    },
)

In [ ]:
# Build an agent with GitHub MCP tools
from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.llms import OllamaLLM

llm = OllamaLLM(model="glm-5.1:cloud")

agent = (
    await LLMAgentBuilder()
    .with_llm(llm)
    .with_mcp_provider(github_mcp_provider)
    .build()
)

# Print all discovered tools
for tool in agent.tools:
    desc = (tool.description or "")[:50]
    print(f"{tool.name}:\n\t{desc}...")

In [ ]:
# Ask our agent to get the latest release of our own repo
from llm_agents_from_scratch.data_structures import Task

instruction = (
    "Use the get_latest_release tool to get the latest release for the "
    "GitHub repo with owner 'nerdai' and repo 'llm-agents-from-scratch'."
)
task = Task(instruction=instruction)
handler = agent.run(task, max_steps=10)

In [ ]:
result = handler.exception() or handler.result()
print(result)

![Slide 17](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/17.svg)

![Slide 18](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/18.svg)

![Slide 19](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/19.svg)

![Slide 20](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/20.svg)

In [1]:
# Discover available skills (scans .agents/skills/ in project directory)
from llm_agents_from_scratch.data_structures.skill import SkillScope
from llm_agents_from_scratch.skills.discovery import discover_skills

skills = discover_skills([SkillScope.PROJECT])
print(f"Discovered {len(skills)} skill(s):\n")
for name, skill in skills.items():
    print(f"  Name:        {name}")
    print(f"  Description: {skill.frontmatter.description}")
    print(f"  Location:    {skill.location}")
    print(f"  Resources:   {skill.resources}")
    print(f"\n  Catalog XML:\n{skill.catalog()}")

Discovered 1 skill(s):

  Name:        hailstone-sequence
  Description: Compute the complete Hailstone sequence for a given starting number by repeatedly calling the next_number tool until the value 1 is reached.
  Location:    /Users/nerdai/Projects/llm-agents-from-scratch/talks/2026/toronto-serverless/.agents/skills/hailstone-sequence/SKILL.md
  Resources:   []

  Catalog XML:
<skill>
    <name>hailstone-sequence</name>
    <description>Compute the complete Hailstone sequence for a given starting number by repeatedly calling the next_number tool until the value 1 is reached.</description>
    <location>/Users/nerdai/Projects/llm-agents-from-scratch/talks/2026/toronto-serverless/.agents/skills/hailstone-sequence/SKILL.md</location>
  </skill>


![Slide 21](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/21.svg)

In [2]:
# Read the skill's body — these are the instructions the agent will follow
skill = skills["hailstone-sequence"]
body = skill.read_body()
print(f"=== Skill: {skill.frontmatter.name} ===\n")
print(body)

=== Skill: hailstone-sequence ===

# Hailstone Sequence

Compute the full Hailstone (Collatz) sequence from a starting number down to 1
using the `next_number` tool.

## Arguments

The user must provide a **starting number** (a positive integer greater than 1).

If no starting number is given, ask the user before proceeding.

## Steps

### 1. Initialize

Set the current number `x` to the starting number provided by the user.

Begin tracking the sequence as a list: `[x]`.

### 2. Call the tool

Call `next_number` with the current value of `x`.

```
next_number(x=<current_number>)
```

**STOP and WAIT** for the tool result before continuing.

### 3. Record the result

Append the returned value to the sequence list.

Set `x` to the returned value.

### 4. Check termination

- If `x == 1`, proceed to Step 5.
- Otherwise, go back to Step 2.

**Important rules:**
- NEVER fabricate or simulate tool call results.
- NEVER make multiple tool calls in a single response.
- ALWAYS wait for the actu

![Slide 22](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/22.svg)

In [ ]:
# Activate the hailstone-sequence skill — the programmatic "slash command"
# The agent already has next_number from cell 24, and discover_skills()
# will find .agents/skills/hailstone-sequence/SKILL.md automatically.

handler = llm_agent.run_with_skill(
    skill_name="hailstone-sequence",
    prompt="Compute the Hailstone sequence starting from 12.",
    max_steps=25,
)

In [ ]:
# Inspect the skill-driven result
result = handler.exception() or handler.result()
print(f"Steps taken: {handler.step_counter}")
print(f"\nResult:\n{result}")

![Slide 23](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/23.svg)

![Slide 24](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/24.svg)

![Slide 25](https://d3ddy8balm3goa.cloudfront.net/talks/2026/toronto-serverless/25.svg)